# Detección de pose y seguimiento multiobjeto de jugadores

**Conjunto de datos:** SoccerNet Tracking 2023 — partición *test*, secuencia `SNMOT-137`
**Proyecto:** Sistema de detección automática de fuera de juego · Fase 1 (semanas 1–2): línea base de percepción

---

## Resumen

Este cuaderno implementa la línea base de percepción del sistema de detección de fuera de juego. En cada fotograma de la secuencia se detectan las personas presentes, se estiman sus 17 puntos clave corporales (esquema COCO) y se asigna a cada jugador un identificador que se mantiene a lo largo del clip, incluso ante oclusiones o cruces entre jugadores. Cada jugador se clasifica además por equipo a partir del color de su camiseta y recibe un único identificador desde que aparece en pantalla, que se mantiene en las disputas y tras ausencias breves gracias a la continuidad de su posición y a su equipo.

Los resultados constituyen la entrada de las fases siguientes del proyecto, empezando por la estimación de la homografía entre la imagen y el terreno de juego.

## Estructura del cuaderno

| Sección | Contenido |
|:---:|---|
| 1 | Configuración del entorno de ejecución |
| 2 | Parámetros, rutas y verificación del almacenamiento |
| 3 | Preparación de los datos de entrada |
| 4 | Validación del script de inferencia |
| 5 | Inferencia: pose, equipos e identidades |
| 6 | Persistencia de los resultados |

## Requisitos de ejecución

- Entorno de Google Colab con GPU.
- Ejecución secuencial de las celdas (*Entorno de ejecución → Ejecutar todas*).
- Al menos 10 GB libres en el disco local de Colab para la descarga del conjunto de datos.

## Estrategia de almacenamiento

El cuaderno separa el almacenamiento de trabajo del almacenamiento persistente:

| Ubicación | Contenido | Persistencia |
|---|---|---|
| Disco local de Colab (`/content`) | Archivo del conjunto de datos (8,7 GB), vídeo de trabajo y resultados intermedios | Se elimina al cerrar la sesión |
| Google Drive | Copia del vídeo de la secuencia, anotaciones de referencia y resultados finales | Permanente |

Las operaciones de entrada/salida intensivas se realizan en el disco local. Google Drive es un sistema de archivos remoto, sujeto a cuota, cuyas lecturas pueden bloquearse bajo carga; por ello se reserva para archivos de tamaño reducido que deben conservarse entre sesiones.

## 1. Configuración del entorno de ejecución

Se monta Google Drive, se establece el repositorio del proyecto como directorio de trabajo y se instalan las dependencias:

| Paquete | Función |
|---|---|
| `requirements.txt` | Dependencias del proyecto |
| `SoccerNet` | Cliente oficial de descarga del conjunto de datos |
| `lap` (≥ 0.5.12) | Resolución del problema de asignación lineal que emplean los *trackers* de Ultralytics para asociar detecciones y trayectorias |

La instalación explícita de `lap` evita que Ultralytics la realice durante la inferencia, lo que genera una advertencia de reinicio del entorno.

In [49]:
from google.colab import drive
import os

drive.mount('/content/drive')

ruta_repo = '/content/drive/MyDrive/football/football'
os.chdir(ruta_repo)
print(f"Directorio de trabajo actual: {os.getcwd()}")

!pip install -r requirements.txt -q
!pip install SoccerNet --upgrade -q
!pip install "lap>=0.5.12" -q   # lo necesita el tracker de Ultralytics; instalarlo aquí evita el aviso de "restart runtime"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de trabajo actual: /content/drive/MyDrive/football/football


## 2. Parámetros, rutas y verificación del almacenamiento

Todos los parámetros y rutas se definen en esta celda: analizar otra secuencia o utilizar otro script solo requiere modificar `SEQ` o `SCRIPT_PATH`.

| Variable | Ubicación | Descripción |
|---|---|---|
| `SEQ` | — | Identificador de la secuencia de SoccerNet |
| `SCRIPT_PATH` | Repositorio | Script de inferencia, relativo a la raíz del repositorio |
| `TRACKER_CFG` | Repositorio | Configuración de BoT-SORT ajustada para retransmisiones de fútbol |
| `zip_path` | Disco local | Archivo de la partición *test* de SoccerNet Tracking 2023 |
| `video_local` | Disco local | Vídeo de trabajo sobre el que se ejecuta la inferencia |
| `output_local` | Disco local | Directorio de salida del script |
| `video_drive` | Drive | Copia de seguridad del vídeo de la secuencia |
| `gt_drive` | Drive | Anotaciones de referencia (*ground truth*) de la secuencia |
| `outputs_drive` | Drive | Destino final de los resultados |

La celda verifica que el script, los módulos `team_classification.py` e `identity_tracking.py` (en la misma carpeta que el script) y la configuración del *tracker* existen en el repositorio. Informa además del espacio libre en Drive y emite una advertencia por debajo de 1 GB, umbral a partir del cual las escrituras en Drive pueden fallar o bloquearse.

In [50]:
# --- Configuración: todas las rutas y parámetros en un solo sitio ---
import shutil

SEQ = "SNMOT-137"
SCRIPT_PATH = "scripts/pose_tracking_baseline.py"
TRACKER_CFG = "configs/botsort_football.yaml"   # BoT-SORT ajustado (new_track_thresh, track_buffer)

# Disco local de Colab: rápido y fiable para el trabajo pesado (se borra al cerrar la sesión).
# Leer el vídeo directamente desde Drive provocaba "Stream timeout ... Failed to open video".
local_dir = "/content/soccernet"
zip_path = os.path.join(local_dir, "tracking-2023", "test.zip")   # 8.7 GB, fuera de tu cuota de Drive
video_local = f"/content/{SEQ}.mp4"
output_local = f"/content/outputs/{SEQ}"

# Drive: solo archivos pequeños que quieres conservar entre sesiones
data_dir = os.path.join(ruta_repo, "data", "raw")
video_drive = os.path.join(data_dir, f"{SEQ}.mp4")
gt_drive = os.path.join(data_dir, f"{SEQ}_gt")
outputs_drive = os.path.join(ruta_repo, "outputs", SEQ)

os.makedirs(local_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# Archivos del repositorio que necesita la inferencia
for requerido in [SCRIPT_PATH, TRACKER_CFG] + [os.path.join(os.path.dirname(SCRIPT_PATH), m)
                                              for m in ("team_classification.py", "identity_tracking.py")]:
    if not os.path.exists(os.path.join(ruta_repo, requerido)):
        print(f"⚠️ Falta {requerido} en el repositorio.")

libre_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Espacio libre en Drive: {libre_gb:.1f} GB")
if libre_gb < 1:
    print("⚠️ Drive casi lleno: las escrituras en Drive pueden fallar o colgarse. "
          "Borra lo que no necesites (p. ej. data/raw/tracking-2023) y vacía la papelera de Drive.")

Espacio libre en Drive: 7.7 GB


In [51]:
# Elegir al azar una secuencia de OTRO partido (después de la celda de configuración)
import configparser, os, random, zipfile

if not os.path.exists(zip_path):   # en una sesión nueva el ZIP no está en disco local: se descarga
    from SoccerNet.Downloader import SoccerNetDownloader
    downloader = SoccerNetDownloader(LocalDirectory=local_dir)
    downloader.password = "s0cc3rn3t"
    downloader.downloadDataTask(task="tracking-2023", split=["test"])

def info_partido(zf, ruta_ini):
    """Campos de gameinfo.ini que identifican el partido (p. ej. gameID)."""
    cp = configparser.ConfigParser(strict=False, interpolation=None)
    cp.read_string(zf.read(ruta_ini).decode("utf-8", errors="ignore"))
    sec = cp[cp.sections()[0]] if cp.sections() else {}
    claves = [k for k in sec if "game" in k and "time" not in k]
    return tuple((k, sec[k]) for k in sorted(claves)) or None

with zipfile.ZipFile(zip_path) as zf:
    inis = {n.split("/")[-2]: n for n in zf.namelist() if n.endswith("gameinfo.ini")}
    partidos = {seq: info_partido(zf, ruta) for seq, ruta in inis.items()}

actual = partidos.get(SEQ)
print(f"Actual: {SEQ}  {actual}")
candidatas = [s for s, p in partidos.items() if s != SEQ and p and p != actual]
if not candidatas:   # si gameinfo.ini no identifica el partido: una secuencia lejana en numeración
    num = lambda s: int(s.split("-")[-1])
    candidatas = [s for s in partidos if abs(num(s) - num(SEQ)) > 20] or [s for s in partidos if s != SEQ]
    print("gameinfo.ini no identifica el partido; se elige una secuencia lejana en numeración.")

SEQ = random.choice(sorted(candidatas))
print(f"Elegida: {SEQ}  {partidos[SEQ]}")

# Rutas que dependen de SEQ
video_local = f"/content/{SEQ}.mp4"
output_local = f"/content/outputs/{SEQ}"
video_drive = os.path.join(data_dir, f"{SEQ}.mp4")
gt_drive = os.path.join(data_dir, f"{SEQ}_gt")
outputs_drive = os.path.join(ruta_repo, "outputs", SEQ)

Actual: SNMOT-137  (('gameid', '8'),)
Elegida: SNMOT-118  (('gameid', '7'),)


## 3. Preparación de los datos de entrada

SoccerNet distribuye cada secuencia como fotogramas JPEG individuales (1920 × 1080 px, 25 fps). Esta celda obtiene un vídeo válido de la secuencia en el disco local siguiendo un orden de prioridad que evita repetir trabajo ya realizado:

1. **Vídeo local existente.** Si el vídeo ya se encuentra en el disco local y es válido, se reutiliza.
2. **Copia en Drive.** Si existe una copia de seguridad en Drive, se transfiere al disco local. La copia secuencial resulta más fiable que la lectura directa desde Drive con OpenCV.
3. **Reconstrucción desde el conjunto de datos.** En ausencia de las anteriores:
   - se descarga la partición *test* (8,7 GB) en el disco local, si no estaba ya disponible;
   - se extraen únicamente los archivos de la secuencia, no la partición completa;
   - se conservan en Drive las anotaciones de referencia (`gt.txt`, `gameinfo.ini`, `seqinfo.ini`), necesarias para la evaluación cuantitativa del seguimiento;
   - se codifican los fotogramas en H.264 a 25 fps (CRF 18, `+faststart`), con una calidad prácticamente sin pérdida visible que preserva el detalle de los jugadores lejanos;
   - se guarda una copia del vídeo en Drive para sesiones posteriores.

La función `video_ok` valida cada vídeo abriéndolo y decodificando un fotograma. Un archivo dañado se descarta y el vídeo se reconstruye automáticamente.

In [52]:
import zipfile
import subprocess
from pathlib import Path

import cv2
from tqdm import tqdm


def video_ok(path) -> bool:
    """El vídeo existe y OpenCV puede abrirlo y leer un frame."""
    if not os.path.exists(path):
        return False
    cap = cv2.VideoCapture(str(path))
    ok = cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0 and cap.read()[0]
    cap.release()
    return bool(ok)


def construir_video_desde_zip() -> None:
    # 1. Descargar el ZIP de SoccerNet en disco local (no en Drive)
    if not os.path.exists(zip_path):
        from SoccerNet.Downloader import SoccerNetDownloader
        downloader = SoccerNetDownloader(LocalDirectory=local_dir)
        downloader.password = "s0cc3rn3t"
        downloader.downloadDataTask(task="tracking-2023", split=["test"])
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"La descarga no dejó el ZIP en {zip_path}")

    # 2. Extraer SOLO la secuencia que necesitamos, no todo el split
    extract_dir = Path("/content/temp_tracking")
    with zipfile.ZipFile(zip_path) as zf:
        miembros = [m for m in zf.namelist() if f"{SEQ}/" in m]
        if not miembros:
            raise FileNotFoundError(f"{SEQ} no está dentro de {zip_path}")
        for m in tqdm(miembros, desc=f"Extrayendo {SEQ}", unit="archivo"):
            zf.extract(m, extract_dir)
    seq_dir = next(extract_dir.rglob(f"{SEQ}/img1")).parent

    # 3. Guardar en Drive las anotaciones (pocos KB) para evaluar el tracking más adelante
    os.makedirs(gt_drive, exist_ok=True)
    for rel in ["gt/gt.txt", "gameinfo.ini", "seqinfo.ini"]:
        if (seq_dir / rel).exists():
            shutil.copy(seq_dir / rel, gt_drive)

    # 4. Imágenes -> vídeo, en disco local (+faststart: metadatos al principio del archivo)
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-framerate", "25", "-start_number", "1",
        "-i", str(seq_dir / "img1" / "%06d.jpg"),
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        video_local,
    ], check=True)
    shutil.rmtree(extract_dir, ignore_errors=True)

    # 5. Copia de seguridad en Drive, para no repetir la descarga en otra sesión
    try:
        shutil.copy(video_local, video_drive)
        print(f"Copia de seguridad en Drive: {video_drive}")
    except OSError as e:
        print(f"⚠️ No se pudo copiar el vídeo a Drive ({e}). El vídeo local sirve para esta sesión.")


if video_ok(video_local):
    print(f"El vídeo ya está en disco local: {video_local}")
else:
    if os.path.exists(video_drive):
        try:
            shutil.copy(video_drive, video_local)   # copia secuencial: más fiable que leer desde Drive con OpenCV
        except OSError as e:
            print(f"⚠️ No se pudo copiar desde Drive ({e}).")
    if video_ok(video_local):
        print(f"Vídeo copiado de Drive a {video_local}")
    else:
        construir_video_desde_zip()
        if not video_ok(video_local):
            raise RuntimeError(f"El vídeo generado no se puede abrir: {video_local}")

print(f"Vídeo listo: {video_local}")

Extrayendo SNMOT-118: 100%|██████████| 758/758 [00:02<00:00, 373.17archivo/s]


Copia de seguridad en Drive: /content/drive/MyDrive/football/football/data/raw/SNMOT-118.mp4
Vídeo listo: /content/SNMOT-118.mp4


## 4. Validación del script de inferencia

Antes de lanzar la inferencia se comprueba que el script compila. Si el archivo contiene delimitadores de bloque de código Markdown (por ejemplo, ```` ```python ````, habituales al copiar código desde otras herramientas), estos se eliminan y se guarda la versión corregida. El archivo solo se modifica cuando no compila en su forma original.

Si el error persiste tras la limpieza, se muestran las últimas líneas del script, ya que la causa más probable es un archivo incompleto.

In [53]:
import re
from pathlib import Path

script_file = Path(ruta_repo) / SCRIPT_PATH
code = script_file.read_text(encoding="utf-8-sig")   # -sig: ignora un BOM si lo hay

def quitar_markdown(texto):
    """Devuelve solo el código del primer bloque ```...```, aunque falte el cierre
    o haya texto antes/después del bloque."""
    lineas = texto.splitlines()
    es_fence = lambda l: l.strip().startswith("```")
    inicio = next((i for i, l in enumerate(lineas) if es_fence(l)), None)
    if inicio is None:
        return texto
    fin = next((i for i in range(inicio + 1, len(lineas)) if es_fence(lineas[i])), len(lineas))
    cuerpo = "\n".join(lineas[inicio + 1:fin])
    cuerpo = re.sub(r"\s*```\s*$", "", cuerpo)       # cierre pegado a la última línea de código
    return cuerpo.rstrip() + "\n"

# Si el script se guardó con etiquetas ```python (p. ej. copiado de un chat), quitarlas.
# Solo se toca el archivo si no compila tal cual.
try:
    compile(code, str(script_file), "exec")
    print("Sintaxis del script OK, no hace falta limpiarlo.")
except SyntaxError:
    limpio = quitar_markdown(code)
    try:
        compile(limpio, str(script_file), "exec")
    except SyntaxError:
        print("Tras quitar las etiquetas sigue sin compilar. Últimas líneas del script:")
        print("\n".join(limpio.splitlines()[-5:]))
        raise
    script_file.write_text(limpio, encoding="utf-8")
    print(f"Etiquetas Markdown eliminadas de {SCRIPT_PATH}. Sintaxis OK.")

Sintaxis del script OK, no hace falta limpiarlo.


## 5. Inferencia: pose, equipos e identidades

El script `pose_tracking_baseline.py` procesa el vídeo en tres etapas:

1. **Detección, pose y seguimiento en línea.** Un modelo YOLO de pose de Ultralytics (por defecto, `yolo26n-pose.pt`) detecta a cada persona y estima sus 17 puntos clave, junto con una puntuación de visibilidad para cada uno. BoT-SORT asocia las detecciones entre fotogramas y compensa el movimiento de la cámara. La configuración `TRACKER_CFG` modifica dos parámetros respecto a la de Ultralytics:
   - `new_track_thresh` (0,25 → 0,4): solo una detección con confianza suficiente puede crear un identificador nuevo.
   - `track_buffer` (30 → 60 fotogramas): una trayectoria perdida se conserva durante 2,4 s antes de descartarse.
2. **Clasificación por equipos** (`team_classification.py`).
   - **Medidas de color.** Para cada detección se mide la camiseta, en las franjas laterales del torso entre hombros y caderas según los puntos clave, y el pantalón, entre caderas y rodillas. Se descarta el césped, y si un jugador más cercano a la cámara tapa la zona, no se mide.
   - **Problema que resuelve.** Un único color de camiseta puede engañar: una camiseta oscura con un dorsal grande del color del rival parece del otro equipo vista de espaldas. Por eso el equipo no se decide por el color más cercano, sino en tres pasos:
     1. Las detecciones se agrupan en varios *aspectos* (camiseta + pantalón): por ejemplo, oscuro de frente, oscuro de espaldas con dorsal claro, claro, claro en sombra, árbitro.
     2. Los aspectos se unen en equipos mediante las trayectorias. Mientras un jugador está aislado, sin nadie cerca con quien el *tracker* pueda confundirlo, todas sus detecciones son de la misma persona y, por tanto, del mismo equipo. Si los jugadores aislados pasan habitualmente del aspecto X al Y al darse la vuelta, X e Y son del mismo equipo, sea cual sea el color del dorsal. La partición en dos equipos es la que menos vínculos de trayectoria corta (corte normalizado). Los colores que ninguna equipación tiene y que las trayectorias no vinculan a los jugadores se clasifican como *no encaja*: árbitros, porteros, cuerpo técnico.
     3. Cada tramo aislado recibe un único equipo, por mayoría ponderada por la confianza de sus aspectos. En los grupos de jugadores, cada detección toma el equipo de su aspecto si este es fiable.

     El método no presupone ningún color para dorsales, patrocinadores ni pantalones. Un jugador al que nunca se ve de frente también se clasifica bien si su aspecto de espaldas se ha visto en otros jugadores de su equipo que sí se dan la vuelta.
   - **Salvaguardas.** La etapa se omite, y el script lo indica, si las camisetas no forman dos grupos con una separación clara (*valley ratio*) o si ninguna partición en dos equipos es respetada por los jugadores aislados (corte superior a 0,6), señal de que los colores responden a cambios de iluminación y no a equipaciones.
3. **Identidades** (`identity_tracking.py`). Cada jugador recibe un único identificador desde que aparece en pantalla. El vídeo se recorre fotograma a fotograma y cada detección se asigna a la identidad cuya posición prevista (con el movimiento de cámara compensado) está más cerca, mediante una asignación óptima uno a uno:
   - una detección cuya camiseta se ha medido como del equipo rival recibe una fuerte penalización, de modo que, cuando atacante y defensor se cruzan, prevalece la asignación coherente con los equipos. Es una penalización y no una prohibición para no romper la identidad de un jugador aislado por una camiseta mal medida en un fotograma;
   - mantener el identificador del *tracker* es siempre lo preferido, incluso tras un salto de la caja o un fotograma sin detecciones; solo una evidencia clara de equipo lo contradice. En grupos la preferencia es menor, para que el movimiento previsto decida un cruce;
   - un jugador que desaparece (oclusión, detecciones perdidas) conserva su identidad durante `--max-gap` segundos y la recupera si reaparece cerca de donde se le esperaba, nunca con una camiseta del equipo rival y solo si no hay otra opción casi igual de plausible;
   - el equipo de cada identidad es la mayoría de las clasificaciones de sus fotogramas, ponderada por su confianza, lo que absorbe los errores aislados.

   Un jugador que permanece fuera de plano más de `--max-gap` segundos recibe un identificador nuevo al volver; para el fuera de juego basta con distinguir correctamente a las personas presentes en cada instante. Los intercambios entre compañeros de equipo no se distinguen por color, pero no afectan a la decisión. `--no-teams` desactiva la clasificación por equipos y las identidades se asignan solo por posición.

**Parámetros de esta ejecución**

| Parámetro | Valor | Justificación |
|---|---|---|
| `--imgsz` | `2560` | Reescala los fotogramas 1,33× para favorecer la detección de jugadores lejanos, con un coste aproximado de 1,8× respecto a 1920 |
| `--conf` | `0.1` | Umbral de detección bajo a propósito: el *tracker* necesita las detecciones débiles para mantener las trayectorias. La creación de identificadores nuevos se controla con `new_track_thresh` |
| `--tracker` | `TRACKER_CFG` | BoT-SORT ajustado para retransmisiones de fútbol |
| `--max-gap` | `3` | Tiempo máximo, en segundos, durante el que un jugador que desaparece conserva su identificador |

**Archivos generados** (en `output_local`)

| Archivo | Contenido | Uso posterior |
|---|---|---|
| `<SEQ>_tracked.mp4` | Vídeo anotado (H.264). Cada jugador aparece con su identificador y su equipo, con un color por equipo: A en cian, B en magenta, *no encaja* en amarillo y sin clasificar en gris. Una leyenda en la esquina superior izquierda muestra, junto a cada equipo, el color de camiseta aprendido | Verificación visual |
| `<SEQ>_keypoints.json` | Por fotograma y jugador: identificador (`track_id`), identificador original del *tracker* (`raw_track_id`), equipo del jugador (`team`), clasificación del fotograma (`team_frame`), descriptor de la camiseta (`torso_color`), color del pantalón (`shorts_color`), aspecto (`appearance_group`), confianza de la etiqueta (`team_weight`), caja, confianza y puntos clave con su puntuación de visibilidad (`null` si es inferior a 0,5). Incluye además los aspectos con su equipo y su confianza, y la lista de correcciones de identidad | Homografía y lógica de fuera de juego |
| `<SEQ>_mot.txt` | Trayectorias finales en formato MOTChallenge | Evaluación frente al *ground truth* con TrackEval |

**Métricas de control.** Al finalizar, el script resume la ejecución:

- **Fotogramas con al menos una detección** y **jugadores seguidos por fotograma:** cobertura del detector.
- **Colores de equipo, salvaguardas y aspectos** con su equipo y su confianza, y el **reparto de las etiquetas** comparando el color más cercano con las etiquetas finales.
- **Cambios de identificador del *tracker* corregidos** y **jugadores recuperados con un identificador nuevo del *tracker*,** cada uno con el instante del vídeo en que se produce, para su verificación visual.
- **Identificadores únicos** del *tracker* y finales, **longitud mediana de trayectoria** y **trayectorias de menos de 1 s**.

Una reducción del número de identificadores no garantiza por sí sola un mejor seguimiento. La validación definitiva requiere comparar las trayectorias con las anotaciones de referencia.

In [54]:
# Entrada y salida en disco local; la celda siguiente copia los resultados a Drive.
# Notas sobre los parámetros:
# - Los frames son 1920x1080. imgsz 2560 reescala x1.33 (~1.8x más cómputo y memoria que 1920).
#   Si hay OOM o va muy lento, prueba --imgsz 1920.
# - --conf bajo a propósito: el tracker usa las detecciones débiles para no perder jugadores tapados.
#   Para limitar los IDs nuevos se usa new_track_thresh en TRACKER_CFG, no --conf.
# - --max-gap: segundos que un jugador que desaparece conserva su ID (si vuelve cerca y con su equipo).
# - Equipos: activos por defecto; --no-teams los desactiva (identidades solo por posición).
# - Opcional: --half (FP16, ~2x más rápido en GPU) y --model yolo26s-pose.pt (más preciso, más lento).
!python "{SCRIPT_PATH}" \
    --source "{video_local}" \
    --output-dir "{output_local}" \
    --imgsz 2560 \
    --conf 0.1 \
    --tracker "{TRACKER_CFG}" \
    --max-gap 3

Procesando vídeo: 100% 750/750 [01:31<00:00,  8.22frame/s]
Dibujando vídeo: 100% 750/750 [00:35<00:00, 21.02frame/s]

Processed 750 frames (1920x1080 @ 25.0fps, imgsz=2560).
Frames with >=1 player detected: 750/750 (100.0%)
Tracked players per frame (mean): 13.8

Teams learnt from the clip: A = Lab(73,1,-5), B = Lab(28,38,24) (main shirt colour of each team; kit colours valley ratio 0.09, lower = better separated)
Team split respected by isolated players: cut 0.02 (0 = no isolated player ever crosses it; must be <= 0.6)
Appearance groups (shirt / shorts) joined into teams through isolated trajectories:
  group 4: team A      2510 detections  shirt Lab(73,1,-5) light 64% dark 6%  shorts Lab(38,-21,22)  confidence 1.00
  group 0: team A       898 detections  shirt Lab(58,1,6) light 36% dark 17%  shorts Lab(62,-0,4)  confidence 0.93
  group 5: team A       843 detections  shirt Lab(74,1,-5) light 65% dark 5%  shorts Lab(63,-10,8)  confidence 1.00
  group 7: team B      3066 detections  sh

## 6. Persistencia de los resultados

Los archivos generados se copian del disco local a `outputs/<SEQ>` en el repositorio de Drive. Si alguna copia falla, por ejemplo por falta de cuota, se indica el archivo afectado y su ruta local, desde donde puede descargarse a través del panel *Archivos* de Colab mientras la sesión permanezca activa.

In [55]:
# Copiar los resultados a Drive
from pathlib import Path

resultados = sorted(Path(output_local).glob("*")) if os.path.isdir(output_local) else []
if not resultados:
    print(f"No hay resultados en {output_local}; revisa la salida de la celda anterior.")
os.makedirs(outputs_drive, exist_ok=True)
for f in resultados:
    try:
        shutil.copy(f, outputs_drive)
        print(f"{f.name} -> {outputs_drive}")
    except OSError as e:
        print(f"⚠️ {f.name} no se pudo copiar a Drive ({e}). Descárgalo desde el panel Archivos: {f}")

SNMOT-118_keypoints.json -> /content/drive/MyDrive/football/football/outputs/SNMOT-118
SNMOT-118_mot.txt -> /content/drive/MyDrive/football/football/outputs/SNMOT-118
SNMOT-118_tracked.mp4 -> /content/drive/MyDrive/football/football/outputs/SNMOT-118
